### Tools

Models can request to call tools that perform tasks such as data fetching from a database, searching the web, or running code. 

Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions. (often a JSON schema)
2. A function or co-routine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]
model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A49A2806E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A49A281400>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from langchain.tools import tool

@tool
def get_weather(place: str) -> str:
    '''Get the weather of specified place'''
    return f"It's sunny in {place}"

model_with_tool = model.bind_tools([get_weather])

In [ ]:
response = model_with_tool.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    if 'type' in tool_call:
        print(f"Type: {tool_call['type']}")

content='' additional_kwargs={'tool_calls': [{'id': '069ntstvv', 'function': {'arguments': '{"place":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 220, 'total_tokens': 234, 'completion_time': 0.051824722, 'completion_tokens_details': None, 'prompt_time': 0.010912814, 'prompt_tokens_details': None, 'queue_time': 0.055067161, 'total_time': 0.062737536}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f0f50-5fe3-7e90-a96e-cb5eb3fb0214-0' tool_calls=[{'name': 'get_weather', 'args': {'place': 'Boston'}, 'id': '069ntstvv', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 14, 'total_tokens': 234}
Tool: get_weather
Tool: {'place': 'Boston'}
Tool: tool_call


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
messages: list[HumanMessage | AIMessage] = [HumanMessage(content="What's the weather like in Boston?")]
response = model_with_tool.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_message = model_with_tool.invoke(messages)

print(final_message.text)

The current weather in Boston is sunny.
